# SVG Preview - Agentic AI Hello World video series

A maintainer tool. It renders every diagram under `videos/` and checks for text that runs off the edge of an SVG.

* Works wherever the repo is cloned - it finds the `videos/` folder automatically.
* New video folders (`05-...`, `06-...`) are picked up without editing this notebook.
* Re-run after editing an SVG to see the change.

**How to use:** attach to any compute (Serverless is fine) and click **Run all**.

In [0]:
import base64, glob, os, re
import xml.etree.ElementTree as ET

def notebook_dir():
    """Folder this notebook lives in (works on classic compute and Serverless)."""
    try:
        path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        return "/Workspace" + os.path.dirname(path)
    except Exception:
        return os.getcwd()   # on recent runtimes the working directory is the notebook's folder

def find_videos_root(start):
    """Walk up from `start` until we find the folder that holds course-map.md (the videos/ folder)."""
    path = os.path.abspath(start)
    while True:
        if os.path.exists(os.path.join(path, "course-map.md")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise FileNotFoundError(f"Could not find the videos/ folder (course-map.md) above {start}")
        path = parent

VIDEOS = find_videos_root(notebook_dir())
FOLDERS = ["assets"] + sorted(
    d for d in os.listdir(VIDEOS) if re.match(r"\d\d-", d) and os.path.isdir(os.path.join(VIDEOS, d))
)

def svgs(folder):
    return sorted(glob.glob(os.path.join(VIDEOS, folder, "**", "*.svg"), recursive=True))

print("videos folder:", VIDEOS)
for folder in FOLDERS:
    print(f"  {folder:28} {len(svgs(folder))} SVGs")

In [0]:
SVG_NS = "{http://www.w3.org/2000/svg}"

def inherited(el, parents, attr, default=None):
    """Read an attribute from the element or its nearest parent (like CSS inheritance)."""
    for e in [el] + parents[::-1]:
        if e.get(attr) is not None:
            return e.get(attr)
    return default

def off_canvas_text(path):
    """Estimate each <text> width and report any that run past the left/right edge.
    An approximation (average character widths), so treat hits as 'please look', not proof."""
    root = ET.parse(path).getroot()
    width = float(root.get("width"))
    problems = []

    def walk(el, parents):
        for child in el:
            if child.tag == SVG_NS + "text":
                text = "".join(child.itertext())
                chain = parents + [el]
                size = float(inherited(child, chain, "font-size", "16"))
                family = inherited(child, chain, "font-family", "") or ""
                char_w = 0.62 if ("Menlo" in family or "mono" in family) else 0.55
                w = len(text) * size * char_w
                x = float(child.get("x", 0))
                anchor = inherited(child, chain, "text-anchor", "start")
                left = x - (w / 2 if anchor == "middle" else w if anchor == "end" else 0)
                if left < 2 or left + w > width - 2:
                    problems.append(f"{text[:60]!r} spans x={left:.0f}..{left + w:.0f}, canvas width {width:.0f}")
            walk(child, parents + [el])

    walk(root, [])
    return problems

total = 0
for folder in FOLDERS:
    for path in svgs(folder):
        try:
            problems = off_canvas_text(path)
        except ET.ParseError as e:
            problems = [f"NOT VALID XML: {e}"]
        for p in problems:
            print(f"{os.path.relpath(path, VIDEOS)}: {p}")
        total += len(problems)
print(f"\nChecked {sum(len(svgs(f)) for f in FOLDERS)} SVGs - {total} possible problem(s).")

In [0]:
def preview(folder):
    """Show every SVG in one folder as an image, with its path as a caption."""
    files = svgs(folder)
    parts = [f"<h2 style='font-family:sans-serif'>{folder} ({len(files)} SVGs)</h2>"]
    for path in files:
        with open(path, "rb") as f:
            data = base64.b64encode(f.read()).decode()
        parts.append(
            "<div style='margin:0 0 28px 0'>"
            f"<div style='font-family:monospace;font-size:13px;color:#555;margin-bottom:6px'>{os.path.relpath(path, VIDEOS)}</div>"
            f"<img src='data:image/svg+xml;base64,{data}' style='max-width:100%;border:1px solid #ddd;border-radius:8px'/>"
            "</div>"
        )
    displayHTML("".join(parts))

# One section per folder: shared assets first, then every video in order
for folder in FOLDERS:
    preview(folder)

In [0]:
# Only want one video? Change the folder name and run this cell.
preview("04-no-memory")